In [1]:
# %%
# imports
import pyomo.environ as pe
import pyomo.opt as po
import math
from itertools import permutations

# %% [markdown]
# # Inputs

# %%
# -------------------------------------------------------------------------
# SETS
# -------------------------------------------------------------------------
factories = ['IJmuiden', 'Segal', 'South Wales']
customer_areas = ['Bochum', 'Boenen', 'Dortmund', 'Gelsenkirchen',
                  'Hagen', 'Iserlohn', 'Neuss', 'Schwerte']
long_bar_types = ['1', '2']
rebar_types = ['A', 'B', 'C']
numOfPeriods = 4
vehicles = [1, 2]   # 2 vehicles per factory

# Parameters needed for steel capacity/fleet capacity calculations
length_lb = {'1': 9, '2': 12}
length_rb = {'A': 2.4, 'B': 3.6, 'C': 4.2}
diameter = {'1': 0.057, '2': 0.057, 'A': 0.057, 'B': 0.057, 'C': 0.057}
density = 7.85

# -------------------------------------------------------------------------
# Steel and Vehicle capacity
# -------------------------------------------------------------------------
steel_capacity = {'IJmuiden': 12, 'Segal': 10, 'South Wales': 28}
vehicle_capacity = 10  # tons per vehicle

# -------------------------------------------------------------------------
# Transportation costs
# -------------------------------------------------------------------------
# Fixed cost per factory per period (if at least one vehicle is used)
f_fleet = {'IJmuiden': 130, 'Segal': 150, 'South Wales': 100}

# Variable cost per km
c_fleet = {'IJmuiden': 3, 'Segal': 3, 'South Wales': 3}

# -------------------------------------------------------------------------
# Demand
# -------------------------------------------------------------------------
demand = {
    # Rebar A
    ('A', 1, 'Bochum'): 2,  ('A', 1, 'Boenen'): 4,  ('A', 1, 'Dortmund'): 2,  ('A', 1, 'Gelsenkirchen'): 5,
    ('A', 1, 'Hagen'): 19,  ('A', 1, 'Iserlohn'): 13, ('A', 1, 'Neuss'): 20, ('A', 1, 'Schwerte'): 4,

    ('A', 2, 'Bochum'): 6,  ('A', 2, 'Boenen'): 8,  ('A', 2, 'Dortmund'): 7,  ('A', 2, 'Gelsenkirchen'): 5,
    ('A', 2, 'Hagen'): 23,  ('A', 2, 'Iserlohn'): 19, ('A', 2, 'Neuss'): 16, ('A', 2, 'Schwerte'): 5,

    ('A', 3, 'Bochum'): 5,  ('A', 3, 'Boenen'): 5,  ('A', 3, 'Dortmund'): 6,  ('A', 3, 'Gelsenkirchen'): 5,
    ('A', 3, 'Hagen'): 25,  ('A', 3, 'Iserlohn'): 17, ('A', 3, 'Neuss'): 14, ('A', 3, 'Schwerte'): 3,

    ('A', 4, 'Bochum'): 3,  ('A', 4, 'Boenen'): 10, ('A', 4, 'Dortmund'): 5,  ('A', 4, 'Gelsenkirchen'): 5,
    ('A', 4, 'Hagen'): 16,  ('A', 4, 'Iserlohn'): 14, ('A', 4, 'Neuss'): 26, ('A', 4, 'Schwerte'): 4,

    # Rebar B
    ('B', 1, 'Bochum'): 4,  ('B', 1, 'Boenen'): 5,  ('B', 1, 'Dortmund'): 4,  ('B', 1, 'Gelsenkirchen'): 9,
    ('B', 1, 'Hagen'): 15,  ('B', 1, 'Iserlohn'): 22, ('B', 1, 'Neuss'): 12, ('B', 1, 'Schwerte'): 2,

    ('B', 2, 'Bochum'): 5,  ('B', 2, 'Boenen'): 8,  ('B', 2, 'Dortmund'): 5,  ('B', 2, 'Gelsenkirchen'): 10,
    ('B', 2, 'Hagen'): 33,  ('B', 2, 'Iserlohn'): 26, ('B', 2, 'Neuss'): 23, ('B', 2, 'Schwerte'): 8,

    ('B', 3, 'Bochum'): 7,  ('B', 3, 'Boenen'): 12, ('B', 3, 'Dortmund'): 8,  ('B', 3, 'Gelsenkirchen'): 6,
    ('B', 3, 'Hagen'): 31,  ('B', 3, 'Iserlohn'): 20, ('B', 3, 'Neuss'): 30, ('B', 3, 'Schwerte'): 2,

    ('B', 4, 'Bochum'): 8,  ('B', 4, 'Boenen'): 13, ('B', 4, 'Dortmund'): 10, ('B', 4, 'Gelsenkirchen'): 6,
    ('B', 4, 'Hagen'): 33,  ('B', 4, 'Iserlohn'): 27, ('B', 4, 'Neuss'): 30, ('B', 4, 'Schwerte'): 6,

    # Rebar C
    ('C', 1, 'Bochum'): 6,  ('C', 1, 'Boenen'): 6,  ('C', 1, 'Dortmund'): 7,  ('C', 1, 'Gelsenkirchen'): 10,
    ('C', 1, 'Hagen'): 12,  ('C', 1, 'Iserlohn'): 14, ('C', 1, 'Neuss'): 22, ('C', 1, 'Schwerte'): 5,

    ('C', 2, 'Bochum'): 7,  ('C', 2, 'Boenen'): 10, ('C', 2, 'Dortmund'): 6,  ('C', 2, 'Gelsenkirchen'): 9,
    ('C', 2, 'Hagen'): 35,  ('C', 2, 'Iserlohn'): 25, ('C', 2, 'Neuss'): 32, ('C', 2, 'Schwerte'): 6,

    ('C', 3, 'Bochum'): 7,  ('C', 3, 'Boenen'): 15, ('C', 3, 'Dortmund'): 4,  ('C', 3, 'Gelsenkirchen'): 9,
    ('C', 3, 'Hagen'): 33,  ('C', 3, 'Iserlohn'): 23, ('C', 3, 'Neuss'): 31, ('C', 3, 'Schwerte'): 7,

    ('C', 4, 'Bochum'): 7,  ('C', 4, 'Boenen'): 12, ('C', 4, 'Dortmund'): 12, ('C', 4, 'Gelsenkirchen'): 10,
    ('C', 4, 'Hagen'): 38,  ('C', 4, 'Iserlohn'): 24, ('C', 4, 'Neuss'): 31, ('C', 4, 'Schwerte'): 2,
}

# -------------------------------------------------------------------------
# Distance between factories and customer areas in km
# -------------------------------------------------------------------------
dist = {
    # From IJmuiden
    ('IJmuiden', 'IJmuiden'): 0,
    ('IJmuiden', 'Segal'): 284,
    ('IJmuiden', 'South Wales'): 826,
    ('IJmuiden', 'Bochum'): 250,
    ('IJmuiden', 'Boenen'): 282,
    ('IJmuiden', 'Dortmund'): 266,
    ('IJmuiden', 'Gelsenkirchen'): 234,
    ('IJmuiden', 'Hagen'): 289,
    ('IJmuiden', 'Iserlohn'): 299,
    ('IJmuiden', 'Neuss'): 259,
    ('IJmuiden', 'Schwerte'): 279,

    # From Segal
    ('Segal', 'IJmuiden'): 284,
    ('Segal', 'Segal'): 0,
    ('Segal', 'South Wales'): 750,
    ('Segal', 'Bochum'): 203,
    ('Segal', 'Boenen'): 242,
    ('Segal', 'Dortmund'): 222,
    ('Segal', 'Gelsenkirchen'): 198,
    ('Segal', 'Hagen'): 206,
    ('Segal', 'Iserlohn'): 226,
    ('Segal', 'Neuss'): 140,
    ('Segal', 'Schwerte'): 216,

    # From South Wales
    ('South Wales', 'IJmuiden'): 826,
    ('South Wales', 'Segal'): 750,
    ('South Wales', 'South Wales'): 0,
    ('South Wales', 'Bochum'): 866,
    ('South Wales', 'Boenen'): 914,
    ('South Wales', 'Dortmund'): 885,
    ('South Wales', 'Gelsenkirchen'): 859,
    ('South Wales', 'Hagen'): 903,
    ('South Wales', 'Iserlohn'): 913,
    ('South Wales', 'Neuss'): 843,
    ('South Wales', 'Schwerte'): 901,

    # From Bochum
    ('Bochum', 'IJmuiden'): 250,
    ('Bochum', 'Segal'): 203,
    ('Bochum', 'South Wales'): 866,
    ('Bochum', 'Bochum'): 0,
    ('Bochum', 'Boenen'): 55,
    ('Bochum', 'Dortmund'): 21,
    ('Bochum', 'Gelsenkirchen'): 19,
    ('Bochum', 'Hagen'): 41,
    ('Bochum', 'Iserlohn'): 51,
    ('Bochum', 'Neuss'): 56,
    ('Bochum', 'Schwerte'): 39,

    # From Boenen
    ('Boenen', 'IJmuiden'): 282,
    ('Boenen', 'Segal'): 242,
    ('Boenen', 'South Wales'): 914,
    ('Boenen', 'Bochum'): 55,
    ('Boenen', 'Boenen'): 0,
    ('Boenen', 'Dortmund'): 34,
    ('Boenen', 'Gelsenkirchen'): 56,
    ('Boenen', 'Hagen'): 44,
    ('Boenen', 'Iserlohn'): 54,
    ('Boenen', 'Neuss'): 102,
    ('Boenen', 'Schwerte'): 32,

    # From Dortmund
    ('Dortmund', 'IJmuiden'): 266,
    ('Dortmund', 'Segal'): 222,
    ('Dortmund', 'South Wales'): 885,
    ('Dortmund', 'Bochum'): 21,
    ('Dortmund', 'Boenen'): 34,
    ('Dortmund', 'Dortmund'): 0,
    ('Dortmund', 'Gelsenkirchen'): 34,
    ('Dortmund', 'Hagen'): 21,
    ('Dortmund', 'Iserlohn'): 36,
    ('Dortmund', 'Neuss'): 75,
    ('Dortmund', 'Schwerte'): 15,

    # From Gelsenkirchen
    ('Gelsenkirchen', 'IJmuiden'): 234,
    ('Gelsenkirchen', 'Segal'): 198,
    ('Gelsenkirchen', 'South Wales'): 859,
    ('Gelsenkirchen', 'Bochum'): 19,
    ('Gelsenkirchen', 'Boenen'): 56,
    ('Gelsenkirchen', 'Dortmund'): 34,
    ('Gelsenkirchen', 'Gelsenkirchen'): 0,
    ('Gelsenkirchen', 'Hagen'): 56,
    ('Gelsenkirchen', 'Iserlohn'): 66,
    ('Gelsenkirchen', 'Neuss'): 62,
    ('Gelsenkirchen', 'Schwerte'): 54,

    # From Hagen
    ('Hagen', 'IJmuiden'): 289,
    ('Hagen', 'Segal'): 206,
    ('Hagen', 'South Wales'): 903,
    ('Hagen', 'Bochum'): 41,
    ('Hagen', 'Boenen'): 44,
    ('Hagen', 'Dortmund'): 21,
    ('Hagen', 'Gelsenkirchen'): 56,
    ('Hagen', 'Hagen'): 0,
    ('Hagen', 'Iserlohn'): 20,
    ('Hagen', 'Neuss'): 66,
    ('Hagen', 'Schwerte'): 14,

    # From Iserlohn
    ('Iserlohn', 'IJmuiden'): 299,
    ('Iserlohn', 'Segal'): 226,
    ('Iserlohn', 'South Wales'): 913,
    ('Iserlohn', 'Bochum'): 51,
    ('Iserlohn', 'Boenen'): 54,
    ('Iserlohn', 'Dortmund'): 36,
    ('Iserlohn', 'Gelsenkirchen'): 66,
    ('Iserlohn', 'Hagen'): 20,
    ('Iserlohn', 'Iserlohn'): 0,
    ('Iserlohn', 'Neuss'): 85,
    ('Iserlohn', 'Schwerte'): 15,

    # From Neuss
    ('Neuss', 'IJmuiden'): 259,
    ('Neuss', 'Segal'): 140,
    ('Neuss', 'South Wales'): 843,
    ('Neuss', 'Bochum'): 56,
    ('Neuss', 'Boenen'): 102,
    ('Neuss', 'Dortmund'): 75,
    ('Neuss', 'Gelsenkirchen'): 62,
    ('Neuss', 'Hagen'): 66,
    ('Neuss', 'Iserlohn'): 85,
    ('Neuss', 'Neuss'): 0,
    ('Neuss', 'Schwerte'): 75,

    # From Schwerte
    ('Schwerte', 'IJmuiden'): 279,
    ('Schwerte', 'Segal'): 216,
    ('Schwerte', 'South Wales'): 901,
    ('Schwerte', 'Bochum'): 39,
    ('Schwerte', 'Boenen'): 32,
    ('Schwerte', 'Dortmund'): 15,
    ('Schwerte', 'Gelsenkirchen'): 54,
    ('Schwerte', 'Hagen'): 14,
    ('Schwerte', 'Iserlohn'): 15,
    ('Schwerte', 'Neuss'): 75,
    ('Schwerte', 'Schwerte'): 0,
}

inv_capacity = {
    'Bochum': 10,
    'Boenen': 7,
    'Dortmund': 12,
    'Gelsenkirchen': 10,
    'Hagen': 12,
    'Iserlohn': 9,
    'Neuss': 8,
    'Schwerte': 5,
}

# %% [markdown]
# # Weight calculations and cutting patterns

# %%
radius = diameter['A'] / 2
weights_lb = {l: math.pi * (radius**2) * length_lb[l] * density for l in long_bar_types}
weights_rb = {r: math.pi * (radius**2) * length_rb[r] * density for r in rebar_types}

def generate_patterns(long_length, rebar_lengths):
    patterns = []
    max_A = int(long_length // rebar_lengths['A'])
    max_B = int(long_length // rebar_lengths['B'])
    max_C = int(long_length // rebar_lengths['C'])

    for a in range(max_A + 1):
        for b in range(max_B + 1):
            for c in range(max_C + 1):
                total_len = a * rebar_lengths['A'] + b * rebar_lengths['B'] + c * rebar_lengths['C']
                if 0 < total_len <= long_length:
                    patterns.append({'A': a, 'B': b, 'C': c})
    return patterns

patterns_1 = generate_patterns(length_lb['1'], length_rb)
patterns_2 = generate_patterns(length_lb['2'], length_rb)

# %% [markdown]
# # Generate all possible routes

# %%
def route_distance(factory, seq, dist):
    """
    Distance of closed route:
    factory -> seq[0] -> seq[1] -> ... -> seq[-1] -> factory
    """
    total = dist[(factory, seq[0])]
    for i in range(len(seq) - 1):
        total += dist[(seq[i], seq[i+1])]
    total += dist[(seq[-1], factory)]
    return total

route_data = {}            # route_data[f][rid] = {'seq': ..., 'customers': ..., 'cost': ...}
route_ids_by_factory = {}  # route_ids_by_factory[f] = [rid1, rid2, ...]

for f in factories:
    route_data[f] = {}
    route_ids_by_factory[f] = []
    rid = 0

    # all ordered routes of length 1 to 8
    for m in range(1, len(customer_areas) + 1):
        for seq in permutations(customer_areas, m):
            route_data[f][rid] = {
                'seq': seq,
                'customers': tuple(seq),
                'cost': route_distance(f, seq, dist)
            }
            route_ids_by_factory[f].append(rid)
            rid += 1

    print(f"Factory {f}: generated {rid} routes")

FR = [(f, r) for f in factories for r in route_ids_by_factory[f]]

# %% [markdown]
# # Model

# %%
model = pe.ConcreteModel(name="Production_Delivery_Planning_RouteBased_2Vehicles_AllRoutes")

# Sets
model.F = pe.Set(initialize=factories)
model.C = pe.Set(initialize=customer_areas)
model.T = pe.Set(initialize=range(1, numOfPeriods + 1))
model.K = pe.Set(initialize=vehicles)
model.L = pe.Set(initialize=long_bar_types)
model.RB = pe.Set(initialize=rebar_types)
model.P1 = pe.Set(initialize=range(len(patterns_1)))
model.P2 = pe.Set(initialize=range(len(patterns_2)))
model.FR = pe.Set(dimen=2, initialize=FR)

# Parameters
model.demand = pe.Param(
    model.RB, model.T, model.C,
    initialize=lambda m, rb, t, c: demand.get((rb, t, c), 0)
)
model.steel_capacity = pe.Param(model.F, initialize=steel_capacity)
model.vehicle_capacity = pe.Param(initialize=vehicle_capacity)
model.f_fleet = pe.Param(model.F, initialize=f_fleet, within=pe.NonNegativeReals)
model.c_fleet = pe.Param(model.F, initialize=c_fleet, within=pe.NonNegativeReals)
model.weights_lb = pe.Param(model.L, initialize=weights_lb)
model.weights_rb = pe.Param(model.RB, initialize=weights_rb)
model.inv_capacity = pe.Param(model.C, initialize=inv_capacity)

def route_cost_init(m, f, r):
    return route_data[f][r]['cost']
model.route_cost = pe.Param(model.FR, initialize=route_cost_init)

# route_visit[f,r,c] = 1 if customer c is on route r of factory f
route_visit_dict = {}
for f in factories:
    for r in route_ids_by_factory[f]:
        customers_in_route = set(route_data[f][r]['customers'])
        for c in customer_areas:
            route_visit_dict[(f, r, c)] = 1 if c in customers_in_route else 0

model.route_visit = pe.Param(model.FR, model.C, initialize=route_visit_dict, within=pe.Binary)

# %% [markdown]
# # Decision Variables

# %%
# Production
model.x1 = pe.Var(model.F, model.P1, model.T, domain=pe.NonNegativeIntegers)
model.x2 = pe.Var(model.F, model.P2, model.T, domain=pe.NonNegativeIntegers)

# Route selection
# lam[f,r,k,t] = 1 if vehicle k of factory f chooses route r in period t
model.lam = pe.Var(model.FR, model.K, model.T, domain=pe.Binary)

# Factory active
# w[f,t] = 1 if factory f is active in period t
model.w = pe.Var(model.F, model.T, domain=pe.Binary)

# Customer inventory
model.I = pe.Var(model.RB, model.C, model.T, domain=pe.NonNegativeIntegers)

# Delivered rebars
# q[f,k,rb,c,t] = number of rebars type rb delivered by vehicle k of factory f to customer c in period t
model.q = pe.Var(model.F, model.K, model.RB, model.C, model.T, domain=pe.NonNegativeIntegers)

# %% [markdown]
# # Objective Function

# %%
def obj_rule(m):
    routing_cost = sum(
        m.lam[f, r, k, t] * m.route_cost[f, r] * m.c_fleet[f]
        for (f, r) in m.FR for k in m.K for t in m.T
    )

    fixed_cost = sum(
        m.w[f, t] * m.f_fleet[f]
        for f in m.F for t in m.T
    )

    return routing_cost + fixed_cost

model.Obj = pe.Objective(rule=obj_rule, sense=pe.minimize)

# %% [markdown]
# # Constraints

# %%
# -------------------------------------------------------------------------
# Each vehicle can select at most one route per period
# -------------------------------------------------------------------------
def one_route_per_vehicle_rule(m, f, k, t):
    return sum(m.lam[f, r, k, t] for r in route_ids_by_factory[f]) <= 1
model.OneRoutePerVehicle = pe.Constraint(model.F, model.K, model.T, rule=one_route_per_vehicle_rule)

# -------------------------------------------------------------------------
# If a route is selected, the factory must be active
# -------------------------------------------------------------------------
def route_implies_factory_active_rule(m, f, r, k, t):
    return m.lam[f, r, k, t] <= m.w[f, t]
model.RouteImpliesFactoryActive = pe.Constraint(model.FR, model.K, model.T, rule=route_implies_factory_active_rule)

# -------------------------------------------------------------------------
# A customer can be visited by at most one vehicle in one period
# (across all factories and all vehicles)
# -------------------------------------------------------------------------
def unique_visit_rule(m, c, t):
    return sum(
        m.route_visit[f, r, c] * m.lam[f, r, k, t]
        for f in m.F for k in m.K for r in route_ids_by_factory[f]
    ) <= 1
model.UniqueVisit = pe.Constraint(model.C, model.T, rule=unique_visit_rule)

# -------------------------------------------------------------------------
# Deliver only to customers on the selected route
# -------------------------------------------------------------------------
def ship_only_if_on_selected_route_rule(m, f, k, rb, c, t):
    max_rb_in_truck = math.floor(vehicle_capacity / weights_rb[rb])
    return m.q[f, k, rb, c, t] <= max_rb_in_truck * sum(
        m.route_visit[f, r, c] * m.lam[f, r, k, t]
        for r in route_ids_by_factory[f]
    )
model.ShipOnlyIfOnSelectedRoute = pe.Constraint(
    model.F, model.K, model.RB, model.C, model.T,
    rule=ship_only_if_on_selected_route_rule
)

# -------------------------------------------------------------------------
# If a customer is visited on the selected route, at least 1 rebar must be delivered
# -------------------------------------------------------------------------
def deliver_to_all_visited_rule(m, f, k, c, t):
    return sum(m.q[f, k, rb, c, t] for rb in m.RB) >= sum(
        m.route_visit[f, r, c] * m.lam[f, r, k, t]
        for r in route_ids_by_factory[f]
    )
model.DeliverToAllVisited = pe.Constraint(
    model.F, model.K, model.C, model.T,
    rule=deliver_to_all_visited_rule
)

# -------------------------------------------------------------------------
# Vehicle capacity per vehicle
# -------------------------------------------------------------------------
def vehicle_cap_rule(m, f, k, t):
    shipped_weight = sum(
        m.q[f, k, rb, c, t] * m.weights_rb[rb]
        for rb in m.RB for c in m.C
    )
    return shipped_weight <= m.vehicle_capacity
model.VehicleCapConstraint = pe.Constraint(model.F, model.K, model.T, rule=vehicle_cap_rule)

# -------------------------------------------------------------------------
# Production must cover total shipments of each rebar type
# -------------------------------------------------------------------------
def link_prod_rule(m, f, rb, t):
    prod1 = sum(patterns_1[p][rb] * m.x1[f, p, t] for p in m.P1)
    prod2 = sum(patterns_2[p][rb] * m.x2[f, p, t] for p in m.P2)
    shipped = sum(m.q[f, k, rb, c, t] for k in m.K for c in m.C)
    return prod1 + prod2 >= shipped
model.LinkProdConstraint = pe.Constraint(model.F, model.RB, model.T, rule=link_prod_rule)

# -------------------------------------------------------------------------
# Inventory balance at customer
# -------------------------------------------------------------------------
def inv_balance_rule(m, rb, c, t):
    inbound = sum(m.q[f, k, rb, c, t] for f in m.F for k in m.K)

    if t == 1:
        return m.I[rb, c, t] == inbound - m.demand[rb, t, c]
    else:
        return m.I[rb, c, t] == m.I[rb, c, t-1] + inbound - m.demand[rb, t, c]
model.InvBalance = pe.Constraint(model.RB, model.C, model.T, rule=inv_balance_rule)

# -------------------------------------------------------------------------
# Inventory capacity at customer
# -------------------------------------------------------------------------
def inv_cap_rule(m, c, t):
    return sum(m.I[rb, c, t] * m.weights_rb[rb] for rb in m.RB) <= m.inv_capacity[c]
model.InvCap = pe.Constraint(model.C, model.T, rule=inv_cap_rule)

# -------------------------------------------------------------------------
# Factory steel capacity if active
# -------------------------------------------------------------------------
def factory_cap_if_active_rule(m, f, t):
    weight_1 = sum(m.x1[f, p, t] * m.weights_lb['1'] for p in m.P1)
    weight_2 = sum(m.x2[f, p, t] * m.weights_lb['2'] for p in m.P2)
    return weight_1 + weight_2 <= m.steel_capacity[f] * m.w[f, t]
model.FactoryCapIfActive = pe.Constraint(model.F, model.T, rule=factory_cap_if_active_rule)

# -------------------------------------------------------------------------
# Optional tightening:
# if factory is inactive, no routes;
# if routes are used, w must become 1
# -------------------------------------------------------------------------
def active_if_any_route_rule(m, f, t):
    return sum(m.lam[f, r, k, t] for r in route_ids_by_factory[f] for k in m.K) <= len(vehicles) * m.w[f, t]
model.ActiveIfAnyRoute = pe.Constraint(model.F, model.T, rule=active_if_any_route_rule)

# %% [markdown]
# # Solve

# %%
solver = po.SolverFactory('gurobi')
result = solver.solve(model, tee=True, options={'TimeLimit': 25200, 'MIPGap': 0.03})

print(f"\nOptimization Status: {result.solver.status}")
print(f"Termination Condition: {result.solver.termination_condition}")

if (result.solver.status == po.SolverStatus.ok or
    result.solver.termination_condition in [po.TerminationCondition.optimal,
                                            po.TerminationCondition.maxTimeLimit,
                                            po.TerminationCondition.feasible]):
    print(f"Total Objective Cost: €{pe.value(model.Obj):.2f}")

# %% [markdown]
# # Print Results

# %%
print("\nOptimal production, delivery and inventory plan (route-based, 2 vehicles/factory)")

for t in model.T:
    print(f"\n================ PERIOD {t} ================")

    for f in model.F:
        if pe.value(model.w[f, t]) > 0.5:
            print(f"\nFactory: {f}")

            # ---------------------------
            # Production
            # ---------------------------
            total_steel_used = 0
            print("  Production:")

            for p in model.P1:
                val = pe.value(model.x1[f, p, t])
                if val is not None and val > 1e-6:
                    bars = int(round(val))
                    total_steel_used += bars * weights_lb['1']
                    print(f"    • {bars}x 9m bars with pattern {patterns_1[p]}")

            for p in model.P2:
                val = pe.value(model.x2[f, p, t])
                if val is not None and val > 1e-6:
                    bars = int(round(val))
                    total_steel_used += bars * weights_lb['2']
                    print(f"    • {bars}x 12m bars with pattern {patterns_2[p]}")

            print(f"    Steel used: {total_steel_used:.2f} / {steel_capacity[f]} tonnes")

            # ---------------------------
            # Vehicle routes
            # ---------------------------
            for k in model.K:
                chosen_routes = [r for r in route_ids_by_factory[f] if pe.value(model.lam[f, r, k, t]) > 0.5]

                if chosen_routes:
                    r = chosen_routes[0]
                    seq = route_data[f][r]['seq']
                    route_str = " -> ".join([f] + list(seq) + [f])

                    print(f"\n  Vehicle {k}:")
                    print(f"    Route: {route_str}")
                    print(f"    Route distance: {route_data[f][r]['cost']} km")

                    total_load = 0
                    print("    Shipments:")

                    for c in model.C:
                        qtyA = int(round(pe.value(model.q[f, k, 'A', c, t])))
                        qtyB = int(round(pe.value(model.q[f, k, 'B', c, t])))
                        qtyC = int(round(pe.value(model.q[f, k, 'C', c, t])))

                        if qtyA + qtyB + qtyC > 0:
                            print(f"      {c}: A={qtyA}, B={qtyB}, C={qtyC}")
                            total_load += (
                                qtyA * weights_rb['A'] +
                                qtyB * weights_rb['B'] +
                                qtyC * weights_rb['C']
                            )

                    print(f"    Vehicle load: {total_load:.2f} / {vehicle_capacity} tonnes")

    # ---------------------------
    # Inventory per customer
    # ---------------------------
    print("\nEnd Inventory (rebars):")

    for c in model.C:
        invA = int(round(pe.value(model.I['A', c, t])))
        invB = int(round(pe.value(model.I['B', c, t])))
        invC = int(round(pe.value(model.I['C', c, t])))

        if invA + invB + invC > 0:
            total_inv_weight = (
                invA * weights_rb['A'] +
                invB * weights_rb['B'] +
                invC * weights_rb['C']
            )

            print(f"  {c}: A={invA}, B={invB}, C={invC} "
                  f"({total_inv_weight:.2f} / {inv_capacity[c]} tonnes)")

print(f"\nTotal Objective Cost: €{pe.value(model.Obj):.2f}")

Factory IJmuiden: generated 109600 routes
Factory Segal: generated 109600 routes
Factory South Wales: generated 109600 routes
Read LP format model from file C:\Users\julie\AppData\Local\Temp\tmpvww3k_0f.pyomo.lp
Reading time = 46.29 seconds
x1: 2631436 rows, 2631504 columns, 102590796 nonzeros
Set parameter TimeLimit to value 25200
Set parameter MIPGap to value 0.03
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  25200
MIPGap  0.03

Optimize a model with 2631436 rows, 2631504 columns and 102590796 nonzeros (Min)
Model fingerprint: 0xe71cc9cc
Model has 2630412 linear objective coefficients
Variable types: 0 continuous, 2631504 integer (2630412 binary)
Coefficient statistics:
  Matrix range     [5e-02, 2e+02]
  Objective range  [1e+02, 7e+03]
  Bounds range   